In [14]:
#Import Packages
import sys, getopt, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, RepeatedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import PauliFeatureMap, RealAmplitudes
from qiskit.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.utils.loss_functions import L2Loss  # Qiskit ML loss, or use PyTorch's
from qiskit.quantum_info import SparsePauliOp

In [15]:
root_folder = 'QNNR_hybrid'
### Globals
# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Fixed feature sizes
NUM_FEATURES = 3
NUM_QUBITS = NUM_FEATURES
NUM_TARGETS = 1

# Quantum circuit parameters
FEATURE_MAP_REPS_LIST = [1]
ANSATZ_REPS_LIST = [1]
ENTANGLEMENT_LIST = ['linear', 'full', 'circular']

# Training hyperparameters
LEARNING_RATE = 0.01
BATCH_SIZE = 30
NUM_EPOCHS = 100  # Adjust as needed

# K-fold cross-validation parameters
N_REPEATS = 1
TEST_SIZE = 1  # Leave-one-out cross-validation (LOOCV) is suggested since the sample size is too small



In [16]:
def get_qnn_torch_model(entangle, feature_map_reps, ansatz_reps):
    ### Feature Map, Ansatz, then QNN Constructor
    # a. Feature Map: Encodes NUM_FEATURES into NUM_QUBITS
    # ParameterVector for input features
    input_params = ParameterVector("x", NUM_FEATURES)

    feature_map_template = PauliFeatureMap(
        feature_dimension=NUM_FEATURES,  # This tells the template how many input parameters it structurally needs
        reps=feature_map_reps,
        entanglement=entangle
    )

    # Assign the *specific* input parameters from the vector to the template's parameter slots
    # This creates a new circuit instance containing parameters ONLY from input_params (size NUM_FEATURES)
    feature_map = feature_map_template.assign_parameters(input_params)
    print(f"Assigned feature map parameters: {feature_map.num_parameters}")

    # Create a template to find out how many parameters it needs structurally
    ansatz_template = RealAmplitudes(NUM_QUBITS, reps=ansatz_reps, entanglement=entangle)
    # ParameterVector for trainable weights - sized based on the template's structural parameters
    num_ansatz_params = ansatz_template.num_parameters  # This was correctly calculated as 12
    weight_params = ParameterVector("θ", num_ansatz_params)

    # Create the ansatz circuit instance by assigning the weight parameters to the template
    ansatz = ansatz_template.assign_parameters(weight_params)
    print(f"Assigned ansatz parameters: {ansatz.num_parameters}")

    # c. Combine into a full quantum circuit
    qc = QuantumCircuit(NUM_QUBITS)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)

    print(f"Total circuit parameters in qc: {qc.num_parameters}")

    # d. Define Observable(s)
    # For a single output, measure the expectation value of Pauli Z on the first qubit
    # The output of EstimatorQNN will be in the range [-1, 1] for Pauli observables
    from qiskit.quantum_info import SparsePauliOp
    observable = SparsePauliOp.from_list([("Z" + "I" * (NUM_QUBITS - 1), 1.0)])
    # If you have multiple qubits and want to combine their measurements, you can define multiple observables
    # or a more complex one. For instance, if NUM_TARGETS > 1 or you want a richer output from QNN:
    # observables = [SparsePauliOp(f"{'I'*i}Z{'I'*(NUM_QUBITS-1-i)}") for i in range(NUM_QUBITS)]
    # This would give NUM_QUBITS outputs from the QNN.

    # --- 3. EstimatorQNN ---
    # Uses Qiskit's Estimator primitive for expectation value computations
    # By default, Estimator uses a local statevector simulator.
    # For real hardware or more advanced simulation, configure the Estimator.
    estimator = Estimator()

    qnn = EstimatorQNN(
        circuit=qc,
        estimator=estimator,
        input_params=input_params,
        weight_params=weight_params,  # Parameters for trainable weights
        observables=observable,  # Observable to measure
        input_gradients=False  # Set to True if you need gradients w.r.t. inputs
    )

    # --- 4. TorchConnector ---
    # Wrap the QNN into a PyTorch module
    initial_weights = 0.01 * (2 * np.random.rand(qnn.num_weights) - 1)
    qnn_torch_model = TorchConnector(qnn, initial_weights=torch.tensor(initial_weights, dtype=torch.float32))

    return qnn_torch_model

In [17]:
class HybridModel(nn.Module):
    def __init__(self, qnn_model):
        super().__init__()
        # Example: Add classical layers if needed
        # self.classical_pre = nn.Linear(NUM_FEATURES, NUM_FEATURES) # If you want to pre-process features
        self.qnn = qnn_model
        # Example: Add classical layers after the QNN
        # self.classical_post = nn.Linear(qnn_model.output_shape[0], NUM_TARGETS) # output_shape[0] is num_observables
        # If qnn_model.output_shape is (1,), then it's 1.

    def forward(self, x):
        # x = self.classical_pre(x) # If using classical_pre
        x = self.qnn(x)
        # x = self.classical_post(x) # If using classical_post
        return x

In [18]:
def prepare_dataset_k_fold(X, y, train_indices, test_indices):
    # Separate train/test split
    X_train_raw, X_test_raw = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    # Separate element column from the actual features
    element_test = X_test_raw[:, 0]
    element_train = X_train_raw[:, 0]

    # Drop the element column (first column)
    X_train = X_train_raw[:, 1:]
    X_test = X_test_raw[:, 1:]

    full_X = np.vstack([X_train, X_test])

    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(full_X)

    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, y_train, X_test_scaled, y_test, element_test, element_train


In [19]:
def get_arguments(argvs):
    _entangle = ''
    _feature_map_reps = ''
    _ansatz_reps = ''
    try:
        opts, args = getopt.getopt(argvs, "h:e:f:a:", ["entangle=", "feature_map_reps=", "ansatz_reps="])
    except getopt.GetoptError:
        print(root_folder + '.py -e <entangle> -f <feature_map_reps> -a <ansatz_reps>')
        sys.exit(2)
    for opt, arg in opts:
        if opt == '-h':
            print(root_folder + '.py -e <entangle> -f <feature_map_reps> -a <ansatz_reps>')
            sys.exit()
        elif opt in ("-e", "--entangle"):
            _entangle = arg
        elif opt in ("-f", "--feature_map_reps"):
            _feature_map_reps = int(arg)
        elif opt in ("-a", "--ansatz_reps"):
            _ansatz_reps = int(arg)
    return _entangle, _feature_map_reps, _ansatz_reps


In [20]:
date = '24_19_25_1'
if not os.path.exists(f'{root_folder}/result'):
    os.makedirs(f'{root_folder}/result')
if not os.path.exists(f'{root_folder}/logs'):
    os.makedirs(f'{root_folder}/logs')

In [21]:
dataset_name = "qml_training-validation-data.csv"
df = pd.read_csv(dataset_name)
display(df.head())
X = df[['Element', 'el_neg', 'B/GPa', 'Volume/A^3']].values
y = df['SFE/mJm^-3'].values
print(df.shape)

,Element,el_neg,B/GPa,Volume/A^3,SFE/mJm^-3
0,Be,1.57,130.0,8.09,23.48
1,Sc,1.36,57.0,25.00,16.16
2,Ti,1.54,110.0,17.60,24.44
3,Co,1.88,180.0,11.00,37.64
4,Zn,1.65,70.0,15.20,20.98


(21, 5)


In [22]:
y_scaler = MinMaxScaler(feature_range=(-1, 1))
y = y_scaler.fit_transform(y.reshape(-1, 1))


In [23]:
print('Total number of data: ', X.shape[0])
rkf = RepeatedKFold(n_splits=X.shape[0] // TEST_SIZE, n_repeats=N_REPEATS)
print(rkf)

Total number of data:  21
RepeatedKFold(n_repeats=1, n_splits=21, random_state=None)


In [24]:
df = pd.DataFrame(columns=['entanglement', 'feature_map_reps', 'ansatz_reps',
                            'element test', 'actual test', 'predicted test',
                            'element train', 'actual train', 'predicted train',
                            'R2 test', 'R2 train'])

In [25]:
# Build output filename
if len(FEATURE_MAP_REPS_LIST) == 1:
        FEATURE_MAP_REPS_LIST_NAME = FEATURE_MAP_REPS_LIST[0]
else:
    FEATURE_MAP_REPS_LIST_NAME = FEATURE_MAP_REPS_LIST
if len(ANSATZ_REPS_LIST) == 1:
    ANSATZ_REPS_LIST_NAME = ANSATZ_REPS_LIST[0]
else:
    ANSATZ_REPS_LIST_NAME = ANSATZ_REPS_LIST
if len(ENTANGLEMENT_LIST) == 1:
    ENTANGLEMENT_LIST_NAME = ENTANGLEMENT_LIST[0]
else:
    ENTANGLEMENT_LIST_NAME = ENTANGLEMENT_LIST
file_name = f'{root_folder}/result/FMR_{FEATURE_MAP_REPS_LIST_NAME}_AR_{ANSATZ_REPS_LIST_NAME}_E_{ENTANGLEMENT_LIST_NAME}_{date}.csv'
print(file_name)


QNNR_hybrid/result/FMR_1_AR_1_E_['linear', 'full', 'circular']_24_19_25_1.csv


In [26]:
i = 0

# LOSS = nn.MSELoss()
LOSS = nn.HuberLoss()  # maybe less prone to extreme values
# LOSS = nn.SmoothL1Loss()
# LOSS = WeightedLoss(small_threshold=5, large_threshold=30, weight_small=16.0, weight_large=4.0)

print("\n--- Start K-Fold Loop ---")

for train_indices, test_indices in rkf.split(X):
    # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, y_train, X_test, y_test, element_test, element_train = prepare_dataset_k_fold(X, y, train_indices, test_indices)

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32)

    # Create DataLoaders
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_dataset = TensorDataset(X_test_t, y_test_t)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"Training data shape: X_train_t: {X_train_t.shape}, y_train_t: {y_train_t.shape}")
    print(f"Testing data shape: X_test_t: {X_test_t.shape}, y_test_t: {y_test_t.shape}")

    # For binary classification (0 or 1 target):
    # You might scale the QNN output (e.g., (output + 1) / 2 to get [0,1]) and then use nn.BCELoss()
    # Or use nn.BCEWithLogitsLoss() if your QNN output is treated as logits (less common for direct EstimatorQNN output).

    for entanglement in ENTANGLEMENT_LIST:
        for feature_map_reps in FEATURE_MAP_REPS_LIST:
            for ansatz_reps in ANSATZ_REPS_LIST:
                # Build model
                # model = qnn_torch_model # for purely quantum nn
                model = HybridModel(get_qnn_torch_model(entangle=entanglement,
                                                        feature_map_reps=feature_map_reps,
                                                        ansatz_reps=ansatz_reps))  # classical modifications in the HybridQNN class
                optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

                print(f"\n--- Starting Training {i}th---")
                train_losses = []
                test_losses = []

                for epoch in range(NUM_EPOCHS):
                    # Training phase
                    model.train()
                    running_loss = 0.0
                    for batch_X, batch_y in train_loader:
                        optimizer.zero_grad()  # Clear gradients
                        outputs = model(batch_X)  # Forward pass
                        loss = LOSS(outputs, batch_y)  # Calculate loss
                        loss.backward()  # Backward pass (compute gradients)
                        optimizer.step()  # Update weights
                        running_loss += loss.item() * batch_X.size(0)

                    epoch_loss = running_loss / len(train_loader.dataset)
                    train_losses.append(epoch_loss)

                    # Validation/Test phase
                    model.eval()
                    test_loss = 0.0
                    with torch.no_grad():  # Disable gradient calculations
                        for batch_X_test, batch_y_test in test_loader:
                            outputs_test = model(batch_X_test)
                            loss_test = LOSS(outputs_test, batch_y_test)
                            test_loss += loss_test.item() * batch_X_test.size(0)

                    epoch_test_loss = test_loss / len(test_loader.dataset)
                    test_losses.append(epoch_test_loss)

                    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Train Loss: {epoch_loss:.4f}, Test Loss: {epoch_test_loss:.4f}")

                print("--- Training Finished ---")

                    # --- 9. Plotting Training History (Optional) ---
                    # plt.figure(figsize=(10, 5))
                    # plt.plot(train_losses, label='Training Loss')
                    # plt.plot(test_losses, label='Test Loss')
                    # plt.title('Training and Test Loss Over Epochs')
                    # plt.xlabel('Epoch')
                    # plt.ylabel('MSE Loss')
                    # plt.legend()
                    # plt.grid(True)
                    # plt.show()

                # --- 10. Evaluation on Test (and Training) Set ---
                model.eval()
                all_preds = []
                all_targets = []
                all_preds_train = []
                all_targets_train = []
                with torch.no_grad():
                    for batch_X_test, batch_y_test in test_loader:
                        outputs_test = model(batch_X_test)
                        all_preds.extend(outputs_test.cpu().numpy())
                        all_targets.extend(batch_y_test.cpu().numpy())
                    for batch_X_train, batch_y_train in train_loader:
                        outputs_train = model(batch_X_train)
                        all_preds_train.extend(outputs_train.cpu().numpy())
                        all_targets_train.extend(batch_y_train.cpu().numpy())

                all_preds = np.array(all_preds)
                all_targets = np.array(all_targets)
                all_preds = y_scaler.inverse_transform(all_preds.reshape(-1, 1))
                all_targets = y_scaler.inverse_transform(all_targets.reshape(-1, 1))

                all_preds_train = np.array(all_preds_train)
                all_targets_train = np.array(all_targets_train)
                all_preds_train = y_scaler.inverse_transform(all_preds_train.reshape(-1, 1))
                all_targets_train = y_scaler.inverse_transform(all_targets_train.reshape(-1, 1))

                    # Example: Scatter plot for regression
                    # if NUM_TARGETS == 1: # Simple plot if single target variable
                    # plt.figure(figsize=(8, 8))
                    # plt.scatter(all_targets, all_preds, alpha=0.5)
                    # plt.plot([min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())],
                    #         [min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())],
                    #         'k--', lw=2, label='Ideal')
                    # plt.xlabel('Actual Values')
                    # plt.ylabel('Predicted Values')
                    # plt.title('Actual vs. Predicted Values on Test Set')
                    # plt.legend()
                    # plt.grid(True)
                    # plt.show()

                    # Further evaluation metrics can be added here (e.g., R-squared for regression, accuracy for classification)
                final_mse = mean_squared_error(all_targets, all_preds)
                final_r2 = r2_score(all_targets, all_preds)
                print(f"\n--- Final Test Set Evaluation ---")
                print(f"Mean Squared Error (MSE): {final_mse:.4f}")
                print(f"R-squared (R2 Score): {final_r2:.4f}")

                final_train_mse = mean_squared_error(all_targets_train, all_preds_train)
                final_train_r2 = r2_score(all_targets_train, all_preds_train)
                print(f"\n--- Final Train Set Evaluation ---")
                print(f"Mean Squared Error (MSE): {final_train_mse:.4f}")
                print(f"R-squared (R2 Score): {final_train_r2:.4f}")

                print(f"\n--- Done for entanglement: {entanglement}, feature_map_reps: {feature_map_reps}, ansatz_reps: {ansatz_reps} ---")

                # Add to dataframe
                new_row = {'entanglement': entanglement,
                            'feature_map_reps': feature_map_reps,
                            'ansatz_reps': ansatz_reps,
                            'element test': element_test,
                            'actual test': np.array(all_targets).flatten(),
                            'predicted test': np.array(all_preds).flatten(),
                            'element train': element_train,
                            'actual train': np.array(all_targets_train).flatten(),
                            'predicted train': np.array(all_preds_train).flatten(),
                            'R2 test': final_r2,
                            'R2 train': final_train_r2,
                            }
                df.loc[len(df)] = new_row
                with np.printoptions(linewidth=10000):
                    df.to_csv(file_name, index=False)  # update csv every loop
                df.at[0, "info"] = [f"DATASET: {dataset_name}, LEARNING_RATE = {LEARNING_RATE}, "
                                    f"BATCH_SIZE = {BATCH_SIZE}, NUM_EPOCHS = {NUM_EPOCHS}, LOSS: {LOSS}"]


--- Start K-Fold Loop ---
Training data shape: X_train_t: torch.Size([20, 3]), y_train_t: torch.Size([20, 1])
Testing data shape: X_test_t: torch.Size([1, 3]), y_test_t: torch.Size([1, 1])
Assigned feature map parameters: 3
Assigned ansatz parameters: 6
Total circuit parameters in qc: 9


/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/qiskit_machine_learning/connectors/torch_connector.py:306: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._weights.data = torch.tensor(initial_weights, dtype=torch.float)



--- Starting Training 0th---
Epoch 1/100, Train Loss: 0.2219, Test Loss: 0.0126
Epoch 2/100, Train Loss: 0.2215, Test Loss: 0.0134
Epoch 3/100, Train Loss: 0.2212, Test Loss: 0.0143
Epoch 4/100, Train Loss: 0.2209, Test Loss: 0.0152
Epoch 5/100, Train Loss: 0.2206, Test Loss: 0.0161
Epoch 6/100, Train Loss: 0.2203, Test Loss: 0.0171
Epoch 7/100, Train Loss: 0.2200, Test Loss: 0.0180
Epoch 8/100, Train Loss: 0.2197, Test Loss: 0.0191
Epoch 9/100, Train Loss: 0.2195, Test Loss: 0.0201
Epoch 10/100, Train Loss: 0.2192, Test Loss: 0.0212
Epoch 11/100, Train Loss: 0.2190, Test Loss: 0.0223
Epoch 12/100, Train Loss: 0.2188, Test Loss: 0.0235
Epoch 13/100, Train Loss: 0.2185, Test Loss: 0.0246
Epoch 14/100, Train Loss: 0.2183, Test Loss: 0.0258
Epoch 15/100, Train Loss: 0.2181, Test Loss: 0.0269
Epoch 16/100, Train Loss: 0.2179, Test Loss: 0.0281
Epoch 17/100, Train Loss: 0.2178, Test Loss: 0.0292
Epoch 18/100, Train Loss: 0.2176, Test Loss: 0.0304
Epoch 19/100, Train Loss: 0.2174, Test Loss

/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/qiskit_machine_learning/connectors/torch_connector.py:306: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._weights.data = torch.tensor(initial_weights, dtype=torch.float)


Epoch 1/100, Train Loss: 0.2213, Test Loss: 0.0094
Epoch 2/100, Train Loss: 0.2190, Test Loss: 0.0077
Epoch 3/100, Train Loss: 0.2167, Test Loss: 0.0063
Epoch 4/100, Train Loss: 0.2145, Test Loss: 0.0049
Epoch 5/100, Train Loss: 0.2123, Test Loss: 0.0038
Epoch 6/100, Train Loss: 0.2103, Test Loss: 0.0028
Epoch 7/100, Train Loss: 0.2083, Test Loss: 0.0019
Epoch 8/100, Train Loss: 0.2064, Test Loss: 0.0012
Epoch 9/100, Train Loss: 0.2045, Test Loss: 0.0007
Epoch 10/100, Train Loss: 0.2028, Test Loss: 0.0003
Epoch 11/100, Train Loss: 0.2011, Test Loss: 0.0001
Epoch 12/100, Train Loss: 0.1995, Test Loss: 0.0000
Epoch 13/100, Train Loss: 0.1979, Test Loss: 0.0000
Epoch 14/100, Train Loss: 0.1964, Test Loss: 0.0002
Epoch 15/100, Train Loss: 0.1950, Test Loss: 0.0005
Epoch 16/100, Train Loss: 0.1937, Test Loss: 0.0010
Epoch 17/100, Train Loss: 0.1924, Test Loss: 0.0015
Epoch 18/100, Train Loss: 0.1911, Test Loss: 0.0021
Epoch 19/100, Train Loss: 0.1899, Test Loss: 0.0029
Epoch 20/100, Train L

...

/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/qiskit_machine_learning/connectors/torch_connector.py:306: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._weights.data = torch.tensor(initial_weights, dtype=torch.float)


Epoch 1/100, Train Loss: 0.2216, Test Loss: 0.0049
Epoch 2/100, Train Loss: 0.2191, Test Loss: 0.0056
Epoch 3/100, Train Loss: 0.2167, Test Loss: 0.0063
Epoch 4/100, Train Loss: 0.2144, Test Loss: 0.0071
Epoch 5/100, Train Loss: 0.2121, Test Loss: 0.0079
Epoch 6/100, Train Loss: 0.2100, Test Loss: 0.0088
Epoch 7/100, Train Loss: 0.2079, Test Loss: 0.0097
Epoch 8/100, Train Loss: 0.2059, Test Loss: 0.0106
Epoch 9/100, Train Loss: 0.2040, Test Loss: 0.0115
Epoch 10/100, Train Loss: 0.2021, Test Loss: 0.0125
Epoch 11/100, Train Loss: 0.2004, Test Loss: 0.0135
Epoch 12/100, Train Loss: 0.1987, Test Loss: 0.0145
Epoch 13/100, Train Loss: 0.1971, Test Loss: 0.0155
Epoch 14/100, Train Loss: 0.1956, Test Loss: 0.0165
Epoch 15/100, Train Loss: 0.1941, Test Loss: 0.0176
Epoch 16/100, Train Loss: 0.1927, Test Loss: 0.0186
Epoch 17/100, Train Loss: 0.1914, Test Loss: 0.0197
Epoch 18/100, Train Loss: 0.1901, Test Loss: 0.0208
Epoch 19/100, Train Loss: 0.1889, Test Loss: 0.0218
Epoch 20/100, Train L

/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/qiskit_machine_learning/connectors/torch_connector.py:306: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._weights.data = torch.tensor(initial_weights, dtype=torch.float)


Epoch 1/100, Train Loss: 0.2214, Test Loss: 0.0051
Epoch 2/100, Train Loss: 0.2171, Test Loss: 0.0058
Epoch 3/100, Train Loss: 0.2130, Test Loss: 0.0066
Epoch 4/100, Train Loss: 0.2090, Test Loss: 0.0074
Epoch 5/100, Train Loss: 0.2052, Test Loss: 0.0083
Epoch 6/100, Train Loss: 0.2015, Test Loss: 0.0092
Epoch 7/100, Train Loss: 0.1980, Test Loss: 0.0101
Epoch 8/100, Train Loss: 0.1947, Test Loss: 0.0110
Epoch 9/100, Train Loss: 0.1916, Test Loss: 0.0120
Epoch 10/100, Train Loss: 0.1886, Test Loss: 0.0130
Epoch 11/100, Train Loss: 0.1858, Test Loss: 0.0140
Epoch 12/100, Train Loss: 0.1831, Test Loss: 0.0150
Epoch 13/100, Train Loss: 0.1806, Test Loss: 0.0160
Epoch 14/100, Train Loss: 0.1783, Test Loss: 0.0170
Epoch 15/100, Train Loss: 0.1761, Test Loss: 0.0181
Epoch 16/100, Train Loss: 0.1740, Test Loss: 0.0191
Epoch 17/100, Train Loss: 0.1721, Test Loss: 0.0201
Epoch 18/100, Train Loss: 0.1703, Test Loss: 0.0212
Epoch 19/100, Train Loss: 0.1687, Test Loss: 0.0222
Epoch 20/100, Train L

/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
